Questions

1. Count the number of actions

2: What are the unique actions recorded in the dataset?

3: Calculate the time duration between consecutive ctivities for each user


data = [

    (1, "login", "2023-08-20 10:23:45"),

    (2, "view", "2023-08-20 11:15:30"),

    (1, "purchase", "2023-08-20 12:45:18"),

    (3, "view", "2023-08-20 13:30:22")

]

columns = ["user_id", "action", "timestamp"]


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, unix_timestamp, lag
from pyspark.sql.window import Window

# Initialize Spark Session
spark = SparkSession.builder.appName("UserActionsAnalysis").getOrCreate()

# Sample dataset
data = [
    (1, "login", "2023-08-20 10:23:45"),
    (2, "view", "2023-08-20 11:15:30"),
    (1, "purchase", "2023-08-20 12:45:18"),
    (3, "view", "2023-08-20 13:30:22")
]
columns = ["user_id", "action", "timestamp"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert timestamp to timestamp type
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

# Question 1: Count the number of actions
total_actions = df.count()
print("Total Actions:", total_actions)

# Question 2: Unique actions recorded in the dataset
unique_actions = df.select("action").distinct()
print("Unique Actions:")
unique_actions.show()

# Question 3: Calculate time duration between consecutive activities for each user
window_spec = Window.partitionBy("user_id").orderBy("timestamp")

# Add a column for previous timestamp per user
df = df.withColumn("prev_timestamp", lag("timestamp").over(window_spec))

# Calculate time difference in seconds
df = df.withColumn("time_diff", (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")).cast("int"))

# Show the results
df.show()


Total Actions: 4
Unique Actions:
+--------+
|  action|
+--------+
|   login|
|    view|
|purchase|
+--------+

+-------+--------+-------------------+-------------------+---------+
|user_id|  action|          timestamp|     prev_timestamp|time_diff|
+-------+--------+-------------------+-------------------+---------+
|      1|   login|2023-08-20 10:23:45|               null|     null|
|      1|purchase|2023-08-20 12:45:18|2023-08-20 10:23:45|     8493|
|      2|    view|2023-08-20 11:15:30|               null|     null|
|      3|    view|2023-08-20 13:30:22|               null|     null|
+-------+--------+-------------------+-------------------+---------+

